# 기술적 분석 Agent · Neon 실데이터 연결
이 노트북만 실행하면 됩니다. 다른 Agent 노트북 실행은 필요하지 않습니다.

**순서:** 패키지 설치 → 설정 → Neon 조회 → 입력 자동 변환 → 품질 표 → 분석 → 선택적 Solar 설명.
기본값은 연결 실행 꺼짐입니다. 실데이터 연결 실패를 예시 데이터로 바꾸지 않습니다.
산업 Agent는 원문 근거 정리의 기본형이며 자동 경쟁력 평가 모델은 아직 아닙니다.
기술적 Agent는 종가·거래량 기본 계산이며 미래 가격 예측 모델은 아닙니다.

실행 흐름: **품질 상태 확인 → 기업별 입력 검사 → Python 계산·근거 연결 → LLM 설명(선택) → Verifier 검사**
조회 상태는 셀 4, 기업별 입력 검사와 Python 분석은 셀 5에서 순서대로 처리합니다.
보류 기업은 계산·LLM을 건너뛰고 Verifier에서 보류 유지 여부를 확인합니다.


## 1. 최초 설치
다음 셀의 맨 앞 #을 지우고 한 번 실행하세요. 설치 후 Kernel → Restart Kernel을 선택합니다.
Solar 연결을 사용하려면 선택 설치 줄도 실행합니다.

In [ ]:
# %pip install -r ../requirements-neon-agents.txt
# %pip install -r ../requirements-basic-agents-solar.txt

## 2. 공통 소스 위치와 설정
압축을 전체 해제하고 notebooks 폴더에서 이 파일을 여세요. .env는 agent 폴더와 나란히 있는 프로젝트 루트에 둡니다.
DATABASE_URL에 읽기 전용 Neon 연결 문자열을 입력합니다. 노트북에는 비밀번호를 쓰지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'agent/neon_agents.py').exists()), None)
if ROOT is None:
    raise RuntimeError('압축 전체를 해제한 프로젝트/notebooks에서 실행하세요.')
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env', override=False)
from agent.basic_agents import AgentSettings, IndustryBusinessAgent, TechnicalAnalysisAgent, SolarLLM
from agent.neon_agents import fetch_reports, execute_report

print('DB 주소 설정 여부:', bool(os.getenv('DATABASE_URL')))  # 실제 주소를 출력하지 않습니다.


In [ ]:
DOMAIN = 'technical_analysis'
USE_NEON = False  # .env 설정 후 True로 변경
TICKERS = ['005930']  # 문자열 유지: 앞자리 0 보존
MARKET = 'KR'
EXPECTED_ROLE = 'agent_reader'  # 실제 읽기 전용 접속 역할과 일치시킵니다.
TODAY = datetime.now(ZoneInfo('Asia/Seoul')).date()
END = TODAY.isoformat()
START = (TODAY - timedelta(days=365)).isoformat()
REFERENCE_DATE = TODAY.isoformat()
SETTINGS = AgentSettings(window=20, literacy='L2', style='U')
MAX_PRICE_AGE_DAYS = 5  # 달력일 기준. 공식 거래일 달력 검사는 별도입니다.
PRICE_METADATA_BY_SOURCE = {}
# 기술적 분석: 실제 source 값에 대해 공급자 문서로 확인한 기준만 입력합니다.
# PRICE_METADATA_BY_SOURCE['실제 source 값'] = {
#     'adjustment': 'unadjusted',  # 또는 split_adjusted / total_return
#     'sourceDescription': '확인한 공급자 문서 URL·기준·확인일'
# }
# 빈 상태이면 가격 조정 기준 미확인으로 기술적 분석을 보류합니다.


## 3. 해당 Agent에 필요한 데이터만 조회·자동 변환
읽기 전용 트랜잭션에서 기업별 데이터를 조회합니다. 실제 DB에는 변경하지 않습니다.
기간은 START~END입니다. 조회 상한 초과·권한 오류는 표에 표시하고 분석을 보류합니다.
산업: financial_quarter + 공시/IR document·chunk. 뉴스 요약은 사업 원문으로 대체하지 않습니다.
기술적: price_daily. 결측치를 0으로 채우거나 결측 행을 임의 제거하지 않습니다.

In [ ]:
reports = []
if USE_NEON:
    reports = fetch_reports(TICKERS, START, END, DOMAIN, market=MARKET,
                            expected_role=EXPECTED_ROLE,
                            price_metadata_by_source=PRICE_METADATA_BY_SOURCE)
    print(f'{len(reports)}개 기업의 실제 DB 입력 변환 완료')
else:
    print('연결 대기: .env 설정 후 USE_NEON=True로 바꾸고 설정 셀부터 다시 실행하세요.')

rows = []
for report in reports:
    for check in report['sourceSnapshot']['loadChecks']:
        rows.append({'기업': report['subject']['name'], '종목': report['subject']['ticker'], **check})
display(pd.DataFrame(rows))


## 4. 품질 검사 후 Python Agent 실행
실행 가능: 현재 자동 검사 통과. 일부 가능: 제한 범위만 실행하고 원문·거래일 등 추가 확인.
보류: 분석·ML/DL·LLM을 호출하지 않습니다. 이유를 보완한 후 조회 셀부터 재실행합니다.
두 Agent의 검사는 독립적입니다. 이 노트북에는 선택한 Agent의 상태만 표시됩니다.

In [ ]:
# ML/DL 연결 위치: predict(payload)를 제공하는 도구를 model_tool에 전달합니다.
# 반환 필드: modelVersion, trainingCutoff, status, output, limitations
agent = TechnicalAnalysisAgent(settings=SETTINGS, llm=None, model_tool=None)
results, quality_rows = [], []
for report in reports:
    gate, result = execute_report(report, agent, DOMAIN, reference_date=REFERENCE_DATE,
                                  max_price_age_days=MAX_PRICE_AGE_DAYS)
    results.append(result)
    quality_rows.append({'기업': report['subject']['name'], 'Agent': DOMAIN, '상태': gate['label'],
                         '이유': '; '.join(gate['reasons']) or '현재 자동 검사 통과',
                         '조회 기간': START + ' ~ ' + END})
display(pd.DataFrame(quality_rows))
for result in results:
    display(pd.DataFrame(result['qualityGate']['checks']))


## 5. 주장·근거·계산 결과 확인
claimId/claimVersion → evidenceId → documentId/documentVersion으로 추적합니다.
해시는 저장된 DB 발췌의 내용 해시입니다. 원본 PDF 파일 해시나 공식 버전으로 간주하지 않습니다.
sourceSnapshot의 재무 값은 원래 정밀도를 보존한 문자열이며 아직 검증된 사업 해석은 아닙니다.
다시 조회하면 새 입력 digest를 확인하세요. 자동 버전 이력 저장은 아직 구현하지 않았습니다.

In [ ]:
for report, result in zip(reports, results):
    print(report['subject']['name'], '분석 상태:', result['status'])
    print('입력 digest:', result['inputDigest'])
    display(pd.DataFrame(result['claims']))
    display(pd.DataFrame(result['calculations']))
    display(pd.DataFrame(result['evidence']))
    display(pd.DataFrame(report['documents']))
    if DOMAIN == 'industry_business_model':
        display(pd.DataFrame(report['sourceSnapshot']['financials']))
        display(pd.DataFrame(result.get('competitionTable', [])))
    display(pd.DataFrame({'남은 한계': result['limitations']}))


## 6. 선택: 실제 Solar 설명 연결
.env의 UPSTAGE_API_KEY를 설정하고 RUN_SOLAR=True로 변경합니다. 이때 근거 자료가 외부 LLM에 전달되고 API 비용이 발생할 수 있습니다.
보류 기업은 호출하지 않습니다. 설명은 미검증 LLM 응답이며 새로운 사실·매수 판단으로 확정하지 않습니다.

In [ ]:
RUN_SOLAR = False
solar_results = []
if RUN_SOLAR and any(r['qualityGate']['status'] != 'withheld' for r in results):
    solar_agent = TechnicalAnalysisAgent(settings=SETTINGS, llm=SolarLLM())
    for report, previous in zip(reports, results):
        if previous['qualityGate']['status'] == 'withheld':
            continue
        gate, result = execute_report(report, solar_agent, DOMAIN, reference_date=REFERENCE_DATE,
                                      max_price_age_days=MAX_PRICE_AGE_DAYS)
        solar_results.append(result)
        print(report['subject']['name'], result['explanationStatus'])
        print(result['explanation'])
else:
    print('Solar 미호출: 실행 설정 또는 품질 상태를 확인하세요.')


## 7. Verifier 검사
Python 결과 또는 Solar 설명이 추가된 결과를 검사합니다. 계산·주장·근거를 같은 코드로 재현하여
입력 버전과 출력 변경을 확인합니다. 독립적인 수학 검증이나 원문 사실 검증은 아닙니다.
failed는 오류 수정 후 재실행, withheld는 자료 보완, review_required는 원문·LLM 의미 검수를 뜻합니다.
자동 검사를 통과하더라도 최종 승인으로 표시하지 않습니다.

In [ ]:
from agent.result_verifier import verify_result

verifications = []
final_results = []
for report, basic_result in zip(reports, results):
    selected = next((r for r in solar_results if
                     r['inputDigest'] == report['digest'] and r['agentType'] == DOMAIN), basic_result)
    verdict = verify_result(report, selected, SETTINGS)
    verifications.append(verdict)
    final_results.append(selected)
    print(report['subject']['name'], 'Verifier:', verdict['status'])
    display(pd.DataFrame(verdict['checks']))


## 8. 선택: 결과 저장
기본값은 저장 안 함입니다. 입력 원문이 포함되므로 private_outputs 파일을 공개 저장소에 올리지 마세요.
DB 재조회 실패 후에는 이전 결과를 사용하지 말고 Kernel을 재시작하여 처음부터 실행하세요.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS and results:
    folder = ROOT / 'private_outputs' / DOMAIN
    folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = folder / f'results_{stamp}.json'
    path.write_text(json.dumps({'reports': reports, 'results': results, 'solarResults': solar_results, 'verifications': verifications},
                               ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    print('저장 완료:', path.name)
